# INFOSYS 722 Iteration 4 — Big Data Analytics Solution (BDAS)

**Final Google Colab / PySpark execution notebook**

This notebook is the orchestration layer for the final Iteration 4 pipeline.  
The modelling logic remains in the submitted `src/` scripts so that the same
code can be executed on Google Colab or AWS EC2.

Final workflow:

1. Step 2 — Data Understanding
2. Step 3 — Data Preparation and D2/D3 reference integration
3. Step 4 — Training-only transformation and feature reduction
4. Steps 5–7 — Model fitting, ranking evaluation and KMeans screening
5. Step 8 — Interpretation, baseline comparison, equity analysis and traceability

Run all cells in order after uploading/unzipping the submission folder.


In [1]:
import os
import sys
import glob

# Locate the project automatically after the submission ZIP is extracted.
data_matches = []
for base in ["/content", "."]:
    data_matches.extend(
        glob.glob(
            os.path.join(base, "**", "data", "brfss2023_diabetes_analysis.csv"),
            recursive=True
        )
    )

if not data_matches:
    raise FileNotFoundError(
        "Could not find data/brfss2023_diabetes_analysis.csv. "
        "Upload and unzip the submission folder first."
    )

project_root = os.path.dirname(os.path.dirname(os.path.abspath(data_matches[0])))
os.chdir(project_root)

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("=" * 72)
print("ITERATION 4 BDAS — PROJECT SETUP")
print("=" * 72)
print("Project root :", project_root)
print("Data file    :", data_matches[0])
print("Source folder:", src_path)

required_source = [
    "src/spark_session.py",
    "src/spark_data.py",
    "src/spark_preparation.py",
    "src/spark_transformation.py",
    "src/spark_model.py",
    "src/spark_evaluation.py",
]

missing = [p for p in required_source if not os.path.exists(p)]

if missing:
    raise FileNotFoundError(
        "Missing required source files: " + ", ".join(missing)
    )

print("Source audit : PASSED")


ITERATION 4 BDAS — PROJECT SETUP
Project root : /home/ubuntu/INFOSYS722-BDAS-Diabetes
Data file    : ./data/brfss2023_diabetes_analysis.csv
Source folder: /home/ubuntu/INFOSYS722-BDAS-Diabetes/src
Source audit : PASSED


In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("PySpark not found. Installing...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyspark"
    ])
else:
    import pyspark
    print("PySpark already installed:", pyspark.__version__)

PySpark already installed: 4.2.0


## Step 2 — Data Understanding

Loads the unchanged D1a analysis table in Spark and prints the row count,
schema and distributed summary statistics used to confirm the BDAS input.


In [3]:
%run src/spark_data.py


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/ubuntu/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/05 22:28:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


STEP 2 - DATA UNDERSTANDING

Number of records:


433323

Schema:
root
 |-- ID: integer (nullable = true)
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- Income: integer (nullable = true)


Summary statistics:


26/10/05 22:28:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------------+-----------------+-----------------+-------------------+-------------------+------------------+------------------+------------------+
|summary|                ID|    Diabetes_binary|            HighBP|           HighChol|         CholCheck|               BMI|            Smoker|             Stroke|HeartDiseaseorAttack|       PhysActivity|  HvyAlcoholConsump|      AnyHealthcare|        NoDocbcCost|           GenHlth|         MentHlth|         PhysHlth|           DiffWalk|                Sex|               Age|         Education|            Income|
+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+

## Step 3 — Data Preparation

Final preparation logic includes:

- removal of the identifier;
- removal of 984 records with a missing target;
- removal of duplicate response profiles before partitioning;
- missingness indicators;
- BMI plausibility treatment;
- executable D2 code-label validation;
- D3 WHO BMI-reference validation and BMI-band construction;
- `RiskTest5` scored from 0–9;
- `RiskTest5_flag = 1` when `RiskTest5 >= 5`;
- `ComorbidityCount`;
- reproducible seeded 70:30 stored train/test partition.

The final prepared output is persisted as
`data/clean_diabetes_step3.parquet`.


In [4]:
%run src/spark_preparation.py


STEP 3 - DATA PREPARATION



1. Original Dataset


Rows : 433323
Cols : 21
root
 |-- ID: integer (nullable = true)
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- Income: integer (nullable = true)


ID removed



2. Target Cleaning
Removed   : 984
Remaining : 432339



3. Duplicate Removal
Removed   : 7391
Remaining : 424948

4. Missing Indicators Created
Indicators:
- Income_missing
- HighChol_missing
- BMI_missing
- HvyAlcoholConsump_missing
- CholCheck_missing
- n_missing_fields


...

+----------------+------+
|ComorbidityCount|count |
+----------------+------+
|NULL            |20741 |
|0               |312162|
|1               |71036 |
|2               |17686 |
|3               |3323  |
+----------------+------+


FINAL DATA PREPARATION AUDIT


Rows    : 424948
Columns : 33

Partition Distribution


+---------+------+
|Partition| count|
+---------+------+
|  Testing|127533|
| Training|297415|
+---------+------+


Target Distribution


+---------------+------+
|Diabetes_binary| count|
+---------------+------+
|              0|354796|
|              1| 70152|
+---------------+------+


Missing Indicator Columns
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields

Final Schema
root
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: inte


Saved:
data/clean_diabetes_step3.parquet

D2 integration status: PASSED
D3 integration status: PASSED
STEP 3 COMPLETED


## Step 4 — Data Transformation

The final transformation stage:

- reads the stored Step 3 partition;
- fits median imputation on **training data only**;
- assembles 31 candidate predictors;
- fits `ChiSqSelector` on training data only and retains 24 predictors;
- fits `StandardScaler` on training data only;
- applies the fitted transformations unchanged to the held-out test set.

Expected persisted outputs include:

- `data/train_step4_transformed.parquet`
- `data/test_step4_transformed.parquet`
- `data/selected_features_step4.txt`


In [5]:
%run src/spark_transformation.py


STEP 4 - DATA TRANSFORMATION

1. Loaded Step 3 Output


Rows : 424948
Cols : 33

2. Partition Check


Training rows : 297415
Testing rows  : 127533

3. Input Features Before Reduction
Feature count : 31
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
NoDocbcCost
GenHlth
MentHlth
PhysHlth
DiffWalk
Sex
Age
Education
Income
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5
RiskTest5_flag
ComorbidityCount



4. Imputation Completed
Training-only imputation model fitted and applied.

5. Raw Feature Vector Created
Vector column: raw_features



6. Feature Reduction Completed
Original feature count : 31
Selected feature count : 24

Selected Feature Indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 18, 20, 23, 24, 25, 26, 27, 28]

Selected Feature Names:
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
GenHlth
MentHlth
PhysHlth
DiffWalk
Age
Education
Income
HighChol_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5



7. Projection / Scaling Completed
Output vector column: features

FINAL STEP 4 AUDIT

Training rows : 297415


Testing rows  : 127533

Columns in transformed training set:
Diabetes_binary
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
NoDocbcCost
GenHlth
MentHlth
PhysHlth
DiffWalk
Sex
Age
Education
Income
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5
RiskTest5_flag
ComorbidityCount
Partition
raw_features
selected_raw_features
features

Selected feature count: 24
Selected features:
- HighBP
- HighChol
- CholCheck
- BMI
- Smoker
- Stroke
- HeartDiseaseorAttack
- PhysActivity
- HvyAlcoholConsump
- AnyHealthcare
- GenHlth
- MentHlth
- PhysHlth
- DiffWalk
- Age
- Education
- Income
- HighChol_missing
- CholCheck_missing
- n_missing_fields
- BMI_band
- MentHlth_band
- PhysHlth_band
- RiskTest5



Saved outputs:
data/train_step4_transformed.parquet
data/test_step4_transformed.parquet
data/selected_features_step4.txt


## Steps 5–7 — Methods, Algorithms and Data Mining

Final supervised portfolio:

- Logistic Regression
- Decision Tree (`maxDepth=4`)
- Random Forest
- Gradient-Boosted Trees (GBT)

Class weights are estimated from the training partition.  
All four classifiers score the same untouched held-out testing population.

KMeans is screened at `k = 4, 5, 6`, with the final value selected using
training-population silhouette score.

Final classifier prediction artefacts use the `predictions_*` naming
convention.


In [6]:
%run src/spark_model.py


STEP 5 / 6 / 7 - METHODS, ALGORITHMS, DATA MINING

1. Loaded Step 4 Outputs


Training rows : 297415
Testing rows  : 127533



2. Class Weights
Negative class count : 248400
Positive class count : 49015
Negative class weight: 0.598661
Positive class weight: 3.033918

3. Baseline Evaluation


26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:29:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

k = 4, silhouette = 0.168544


k = 5, silhouette = 0.194987


k = 6, silhouette = 0.164577

Best k selected: 5
Best silhouette: 0.194987

6. Final KMeans Fit and Cluster Profiling


+-------+------+-----------------+------------------+-----------------+------------------+-----------------+---------------------+--------------------+
|cluster|n     |Age_mean         |GenHlth_mean      |Income_mean      |BMI_mean          |RiskTest5_mean   |ComorbidityCount_mean|Positive_Prevalence |
+-------+------+-----------------+------------------+-----------------+------------------+-----------------+---------------------+--------------------+
|0      |41142 |8.97695785328861 |3.920130280491955 |5.670409800204171|30.732778669000012|5.218292742209907|0.9295853385834427   |0.34264255505323027 |
|1      |110996|6.370238567155573|2.1363382464232945|7.684646293560128|26.144048704456836|2.868661933763379|0.03798335075137843  |0.053209124653140656|
|2      |18020 |5.453773584905661|2.6501664816870143|6.557824639289678|27.063371254162476|3.147891231964484|0.18829078801331853  |0.08962264150943396 |
|3      |101970|9.887594390507012|2.6929292929292927|6.896391095420221|30.477701186623  

Saved: data/kmeans_cluster_profile.csv
Saved: data/kmeans_screening_results.csv

FINAL SUMMARY

Classification outputs saved:
- data/predictions_LogisticRegression.parquet
- data/predictions_DecisionTree.parquet
- data/predictions_RandomForest.parquet
- data/predictions_GBT.parquet
- data/model_metrics_step7.csv

Clustering outputs saved:
- data/kmeans_screening_results.csv
- data/kmeans_cluster_profile.csv

Done.


## Step 8 — Interpretation and Evaluation

The final evaluation reloads the latest `predictions_*.parquet` artefacts and
recomputes report-facing evidence, including:

- confusion matrices;
- AUC, accuracy, precision, recall and F1;
- top-20 capture and lift;
- contact depth required to capture 60% of true cases;
- contacts per true case;
- corrected `RiskTest5` baseline comparison;
- BO4 income-band recall analysis;
- KMeans `k=4/5/6` screening and final cluster profiles;
- final 127,533-row traceability check for every classifier.


In [7]:
%run src/spark_evaluation.py


STEP 8 - INTERPRETATION AND EVALUATION

Loading: data/predictions_LogisticRegression.parquet


Logistic Regression

Test rows: 127533



Confusion Matrix


+---------------+----------+-----+
|Diabetes_binary|prediction|count|
+---------------+----------+-----+
|              0|       0.0|75611|
|              0|       1.0|30785|
|              1|       0.0| 5364|
|              1|       1.0|15773|
+---------------+----------+-----+




Classification Metrics
AUC       : 0.803739
Accuracy  : 0.716552
Precision : 0.338782
Recall    : 0.746227
F1        : 0.466002


26/10/05 22:33:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 22:33:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

+------+---------------+------+-----------+-----------+------------------+-------------------+------------------+--------------------+--------------------+
|Income|Actual_Positive|GBT_TP|RiskTest_TP|Logistic_TP|RiskTest_Recall   |Logistic_Recall    |GBT_Recall        |GBT_vs_RiskTest     |Logistic_vs_RiskTest|
+------+---------------+------+-----------+-----------+------------------+-------------------+------------------+--------------------+--------------------+
|1     |641            |546   |420        |556        |0.6552262090483619|0.8673946957878315 |0.8517940717628705|0.19656786271450855 |0.21216848673946953 |
|2     |853            |784   |648        |773        |0.7596717467760844|0.9062133645955451 |0.9191090269636577|0.1594372801875733  |0.14654161781946073 |
|3     |1002           |894   |753        |882        |0.7514970059880239|0.8802395209580839 |0.8922155688622755|0.14071856287425155 |0.12874251497005995 |
|4     |1337           |1177  |978        |1133       |0.7314884


Saved: data/bo4_equity_comparison


KMEANS EVALUATION

KMeans screening results
+---+-------------------+
|k  |silhouette         |
+---+-------------------+
|4  |0.16854421806623268|
|5  |0.19498717081262967|
|6  |0.1645769485041897 |
+---+-------------------+



Best k: 5
Best silhouette: 0.19498717081262967

Final KMeans cluster profile
+-------+------+-----------------+------------------+-----------------+------------------+-----------------+---------------------+--------------------+
|cluster|n     |Age_mean         |GenHlth_mean      |Income_mean      |BMI_mean          |RiskTest5_mean   |ComorbidityCount_mean|Positive_Prevalence |
+-------+------+-----------------+------------------+-----------------+------------------+-----------------+---------------------+--------------------+
|0      |41142 |8.97695785328861 |3.920130280491955 |5.670409800204171|30.732778669000012|5.218292742209907|0.9295853385834427   |0.34264255505323027 |
|1      |110996|6.370238567155573|2.1363382464232945|7.684646293560128|26.144048704456836|2.868661933763379|0.03798335075137843  |0.053209124653140656|
|2      |18020 |5.453773584905661|2.6501664816870143|6.557824639289678|27.063371254162476|3.147891231964484|0.18829078801331853  |0.08962264150943396 |
|3      |10

Logistic Regression: 127,533 rows [OK]
Decision Tree: 127,533 rows [OK]


Random Forest: 127,533 rows [OK]
GBT: 127,533 rows [OK]

Step 8 outputs:
- data/final_model_comparison
- data/risktest5_baseline_step8
- data/bo4_equity_comparison
- data/kmeans_screening_results.csv
- data/kmeans_cluster_profile.csv

STEP 8 COMPLETED


## Final Output Audit

This cell checks that the report-aligned artefacts from the final pipeline
exist. It deliberately checks the current `predictions_*` outputs rather than
the obsolete `result_*` artefacts from earlier development runs.


In [8]:
import os

required_outputs = [
    # Step 3
    "data/clean_diabetes_step3.parquet",

    # Step 4
    "data/train_step4_transformed.parquet",
    "data/test_step4_transformed.parquet",
    "data/selected_features_step4.txt",

    # Step 7 classifier outputs
    "data/predictions_LogisticRegression.parquet",
    "data/predictions_DecisionTree.parquet",
    "data/predictions_RandomForest.parquet",
    "data/predictions_GBT.parquet",
    "data/model_metrics_step7.csv",

    # KMeans
    "data/kmeans_screening_results.csv",
    "data/kmeans_cluster_profile.csv",

    # Step 8
    "data/final_model_comparison",
    "data/risktest5_baseline_step8",
    "data/bo4_equity_comparison",
]

print("=" * 72)
print("FINAL OUTPUT AUDIT")
print("=" * 72)

missing = []

for path in required_outputs:
    if os.path.exists(path):
        print("FOUND  :", path)
    else:
        print("MISSING:", path)
        missing.append(path)

print("=" * 72)

if missing:
    print("FINAL AUDIT WARNING")
    print(f"{len(missing)} expected artefact(s) were not found.")
else:
    print("FINAL AUDIT PASSED")
    print("All report-aligned Iteration 4 artefacts were found.")

print("=" * 72)
print("ITERATION 4 BDAS PIPELINE COMPLETED")
print("=" * 72)


FINAL OUTPUT AUDIT
FOUND  : data/clean_diabetes_step3.parquet
FOUND  : data/train_step4_transformed.parquet
FOUND  : data/test_step4_transformed.parquet
FOUND  : data/selected_features_step4.txt
FOUND  : data/predictions_LogisticRegression.parquet
FOUND  : data/predictions_DecisionTree.parquet
FOUND  : data/predictions_RandomForest.parquet
FOUND  : data/predictions_GBT.parquet
FOUND  : data/model_metrics_step7.csv
FOUND  : data/kmeans_screening_results.csv
FOUND  : data/kmeans_cluster_profile.csv
FOUND  : data/final_model_comparison
FOUND  : data/risktest5_baseline_step8
FOUND  : data/bo4_equity_comparison
FINAL AUDIT PASSED
All report-aligned Iteration 4 artefacts were found.
ITERATION 4 BDAS PIPELINE COMPLETED
